In [0]:
# Databricks notebook: 09_map_merchants
# Purpose: Transform PhonePe map merchant JSON into an
# analytics-ready state/district-level Silver Delta table.

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    MapType,
    StringType,
    BooleanType,
    LongType
)

# ============================================================
# 1. CONFIGURATION
# ============================================================

BRONZE_TABLE = "workspace.bronze.phonepe_raw_json"
METADATA_TABLE = "workspace.silver.phonepe_file_metadata"
SILVER_TABLE = "workspace.silver.map_merchants"

EXPECTED_COUNTRY_FILES = 34
EXPECTED_STATE_FILES = 1_178
EXPECTED_SOURCE_FILES = 1_212

print("Configuration")
print(f"Bronze source : {BRONZE_TABLE}")
print(f"Metadata source: {METADATA_TABLE}")
print(f"Silver target : {SILVER_TABLE}")
print(f"Expected files: {EXPECTED_SOURCE_FILES:,}")


# ============================================================
# 2. SELECT MAP MERCHANT SOURCE FILES
# ============================================================

bronze_df = spark.table(BRONZE_TABLE).alias("b")

map_merchant_metadata_df = (
    spark.table(METADATA_TABLE)
    .filter(
        (F.col("dataset_group") == "map") &
        (F.col("domain") == "merchant")
    )
    .alias("m")
)

map_merchant_source_df = (
    map_merchant_metadata_df
    .join(
        bronze_df,
        F.col("m.source_path") == F.col("b.source_path"),
        "inner"
    )
    .select(
        F.col("m.relative_path"),
        F.col("m.file_scope"),
        F.col("m.state_slug"),
        F.col("m.year"),
        F.col("m.quarter"),
        F.col("m.period"),
        F.col("b.raw_json"),
        F.col("b.source_path"),
        F.col("b.source_commit"),
        F.col("b.content_sha256"),
        F.col("b.ingested_at_utc")
    )
)

source_scope_summary = (
    map_merchant_source_df
    .groupBy("file_scope")
    .agg(
        F.countDistinct("source_path").alias("file_count")
    )
)

source_counts = {
    row["file_scope"]: row["file_count"]
    for row in source_scope_summary.collect()
}

actual_country_files = source_counts.get("country", 0)
actual_state_files = source_counts.get("state", 0)
actual_source_files = actual_country_files + actual_state_files

print("\nSource validation")
print(f"Country files: {actual_country_files:,}")
print(f"State files  : {actual_state_files:,}")
print(f"Total files  : {actual_source_files:,}")

assert actual_country_files == EXPECTED_COUNTRY_FILES
assert actual_state_files == EXPECTED_STATE_FILES
assert actual_source_files == EXPECTED_SOURCE_FILES

print("Map merchant source validation passed.")


# ============================================================
# 3. DEFINE EXPLICIT JSON SCHEMA
# ============================================================

merchant_area_metrics_schema = StructType([
    StructField("registeredCount", LongType(), True),

    # Included for possible future schema evolution.
    StructField("registeredMerchants", LongType(), True)
])

map_merchant_schema = StructType([
    StructField("success", BooleanType(), True),
    StructField("code", StringType(), True),

    StructField(
        "data",
        StructType([
            StructField(
                "hoverData",
                MapType(
                    StringType(),
                    merchant_area_metrics_schema,
                    True
                ),
                True
            )
        ]),
        True
    ),

    StructField("responseTimestamp", LongType(), True)
])

print("\nExplicit map merchant schema created.")


# ============================================================
# 4. PARSE JSON
# ============================================================

parsed_map_merchant_df = (
    map_merchant_source_df
    .withColumn(
        "parsed_json",
        F.from_json(
            F.col("raw_json"),
            map_merchant_schema
        )
    )
)

parse_summary = (
    parsed_map_merchant_df
    .agg(
        F.count("*").alias("source_files"),

        F.sum(
            F.when(
                F.col("parsed_json.data.hoverData").isNull(),
                1
            ).otherwise(0)
        ).alias("files_without_map_data")
    )
    .first()
)

print("\nJSON parsing")
print(f"Source files          : {parse_summary['source_files']:,}")
print(
    f"Files without map data: "
    f"{parse_summary['files_without_map_data']:,}"
)

assert parse_summary["source_files"] == EXPECTED_SOURCE_FILES

assert parse_summary["files_without_map_data"] == 0, (
    "One or more files could not be parsed or do not "
    "contain merchant hoverData."
)

print("Map merchant JSON parsing passed.")


# ============================================================
# 5. FLATTEN DYNAMIC GEOGRAPHIC MAP
# ============================================================

map_merchants_df = (
    parsed_map_merchant_df

    .withColumn(
        "area_entry",
        F.explode(
            F.map_entries(
                F.col("parsed_json.data.hoverData")
            )
        )
    )

    .select(
        F.col("file_scope"),

        F.when(
            F.col("file_scope") == "country",
            F.lit("state")
        )
        .otherwise(F.lit("district"))
        .alias("geography_level"),

        F.when(
            F.col("file_scope") == "state",
            F.col("state_slug")
        )
        .otherwise(F.lit(None).cast("string"))
        .alias("parent_state_slug"),

        F.trim(
            F.col("area_entry.key")
        ).alias("area_name"),

        F.lower(
            F.regexp_replace(
                F.trim(F.col("area_entry.key")),
                "[^a-zA-Z0-9]+",
                "-"
            )
        ).alias("area_slug"),

        F.col("year").cast("int").alias("year"),
        F.col("quarter").cast("int").alias("quarter"),
        F.col("period"),

        F.coalesce(
            F.col("area_entry.value.registeredCount"),
            F.col("area_entry.value.registeredMerchants")
        )
        .cast("long")
        .alias("registered_merchant_count"),

        F.when(
            F.col("area_entry.value.registeredCount").isNotNull(),
            F.lit("registeredCount")
        )
        .when(
            F.col(
                "area_entry.value.registeredMerchants"
            ).isNotNull(),
            F.lit("registeredMerchants")
        )
        .otherwise(F.lit(None))
        .alias("registered_count_source_field"),

        F.col("parsed_json.responseTimestamp")
            .alias("response_timestamp_epoch_ms"),

        F.col("relative_path"),
        F.col("source_path"),
        F.col("source_commit"),
        F.col("content_sha256"),
        F.col("ingested_at_utc")
    )
)

print("\nDynamic geographic merchant records flattened.")


# ============================================================
# 6. DATA-QUALITY VALIDATION
# ============================================================

quality_result = (
    map_merchants_df
    .agg(
        F.count("*").alias("total_rows"),

        F.sum(
            F.when(
                F.col("area_name").isNull() |
                F.col("geography_level").isNull() |
                F.col("registered_merchant_count").isNull(),
                1
            ).otherwise(0)
        ).alias("missing_required_values"),

        F.sum(
            F.when(
                F.col("registered_merchant_count") < 0,
                1
            ).otherwise(0)
        ).alias("negative_merchant_counts"),

        F.countDistinct("source_path")
            .alias("represented_source_files"),

        F.min("registered_merchant_count")
            .alias("minimum_merchant_count"),

        F.max("registered_merchant_count")
            .alias("maximum_merchant_count")
    )
    .first()
)

duplicate_business_keys = (
    map_merchants_df
    .groupBy(
        "source_path",
        "area_name"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("\nData-quality results")
print(f"Total Silver rows        : {quality_result['total_rows']:,}")
print(
    f"Missing required values  : "
    f"{quality_result['missing_required_values']:,}"
)
print(
    f"Negative merchant counts : "
    f"{quality_result['negative_merchant_counts']:,}"
)
print(
    f"Source files represented : "
    f"{quality_result['represented_source_files']:,}"
)
print(
    f"Minimum merchant count   : "
    f"{quality_result['minimum_merchant_count']:,}"
)
print(
    f"Maximum merchant count   : "
    f"{quality_result['maximum_merchant_count']:,}"
)
print(f"Duplicate business keys  : {duplicate_business_keys:,}")

assert quality_result["total_rows"] > 0
assert quality_result["missing_required_values"] == 0
assert quality_result["negative_merchant_counts"] == 0
assert quality_result["represented_source_files"] == EXPECTED_SOURCE_FILES
assert duplicate_business_keys == 0

print("Map merchant quality checks passed.")


# ============================================================
# 7. DISPLAY COVERAGE
# ============================================================

print("\nRows by geography level and source field")

display(
    map_merchants_df
    .groupBy(
        "geography_level",
        "registered_count_source_field"
    )
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("source_path").alias("source_file_count")
    )
    .orderBy("geography_level")
)


# ============================================================
# 8. SAVE MANAGED DELTA TABLE
# ============================================================

(
    map_merchants_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"\nSilver Delta table created: {SILVER_TABLE}")


# ============================================================
# 9. VALIDATE SAVED TABLE
# ============================================================

saved_map_merchants_df = spark.table(SILVER_TABLE)

saved_summary = (
    saved_map_merchants_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("source_path").alias("source_file_count"),
        F.min("period").alias("first_period"),
        F.max("period").alias("last_period")
    )
    .first()
)

print("\nSaved-table validation")
print(f"Saved rows              : {saved_summary['row_count']:,}")
print(f"Source files represented: {saved_summary['source_file_count']:,}")
print(f"First period            : {saved_summary['first_period']}")
print(f"Last period             : {saved_summary['last_period']}")

assert saved_summary["row_count"] > 0
assert saved_summary["source_file_count"] == EXPECTED_SOURCE_FILES
assert saved_summary["first_period"] == "2018-Q1"
assert saved_summary["last_period"] == "2026-Q2"

print("Final map merchant Delta-table validation passed.")


# ============================================================
# 10. DISPLAY SAMPLE
# ============================================================

display(
    saved_map_merchants_df
    .select(
        "geography_level",
        "parent_state_slug",
        "area_name",
        "year",
        "quarter",
        "period",
        "registered_merchant_count",
        "registered_count_source_field"
    )
    .orderBy(
        "year",
        "quarter",
        "geography_level",
        "parent_state_slug",
        "area_name"
    )
    .limit(30)
)

Configuration
Bronze source : workspace.bronze.phonepe_raw_json
Metadata source: workspace.silver.phonepe_file_metadata
Silver target : workspace.silver.map_merchants
Expected files: 1,212

Source validation
Country files: 34
State files  : 1,178
Total files  : 1,212
Map merchant source validation passed.

Explicit map merchant schema created.

JSON parsing
Source files          : 1,212
Files without map data: 0
Map merchant JSON parsing passed.

Dynamic geographic merchant records flattened.

Data-quality results
Total Silver rows        : 25,915
Missing required values  : 0
Negative merchant counts : 0
Source files represented : 1,212
Minimum merchant count   : 1
Maximum merchant count   : 7,866,204
Duplicate business keys  : 0
Map merchant quality checks passed.

Rows by geography level and source field


geography_level,registered_count_source_field,row_count,source_file_count
district,registeredCount,24737,1178
state,registeredCount,1178,34



Silver Delta table created: workspace.silver.map_merchants

Saved-table validation
Saved rows              : 25,915
Source files represented: 1,212
First period            : 2018-Q1
Last period             : 2026-Q2
Final map merchant Delta-table validation passed.


geography_level,parent_state_slug,area_name,year,quarter,period,registered_merchant_count,registered_count_source_field
district,andhra-pradesh,alluri sitharama raju district,2018,1,2018-Q1,4,registeredCount
district,andhra-pradesh,annamayya district,2018,1,2018-Q1,1,registeredCount
district,andhra-pradesh,chittoor district,2018,1,2018-Q1,4,registeredCount
district,andhra-pradesh,guntur district,2018,1,2018-Q1,1,registeredCount
district,andhra-pradesh,kakinada district,2018,1,2018-Q1,2,registeredCount
district,andhra-pradesh,kurnool district,2018,1,2018-Q1,1,registeredCount
district,andhra-pradesh,ntr district,2018,1,2018-Q1,9,registeredCount
district,andhra-pradesh,sri sathyasai district,2018,1,2018-Q1,3,registeredCount
district,andhra-pradesh,tirupati district,2018,1,2018-Q1,2,registeredCount
district,andhra-pradesh,visakhapatnam district,2018,1,2018-Q1,2,registeredCount


# Silver Map Merchants — Completion Summary

The `workspace.silver.map_merchants` Delta table was successfully created and validated.

## Results

- Geographic merchant records: **25,915**
- Source files represented: **1,212**
- Data coverage: **2018-Q1 to 2026-Q2**
- Missing required values: **0**
- Negative merchant counts: **0**
- Duplicate business keys: **0**

## Table grain

Each record represents one geographical area and quarter:

- Country files produce state-level records.
- State files produce district-level records.

## Available metric

- Cumulative registered-merchant count

## Analytical interpretation

Registered-merchant count is cumulative and must not be summed across quarters.

This table supports:

- Merchant-infrastructure density analysis
- State and district merchant growth
- Transactions per merchant
- Transaction value per merchant
- Users per merchant
- Regional digital-acceptance clustering
- Underserved-market identification